# 第四次作业：重力四子棋 AI · Jupyter Notebook 版

本篇接着课堂演示，重点学习 **网络 → 搜索 → 自我对弈数据 → 更新网络** 的循环。Notebook 操作与 PyTorch 基础不再展开。

需要补全：四个搜索步骤，以及对弈记录、胜负标签和训练损失。网络与辅助工具已提供，**只改本笔记本，不必重复填写 `.py` 模板。**

- 用 Remote SSH 打开 `第四次作业-Jupyter Notebook版` 文件夹，保留 `starter/`，选择 `pubpy` 内核。
- 按顺序运行；修改函数后重新运行定义格。补全后删除对应的 `raise NotImplementedError(...)`。
- 检查可直接运行；试下、训练、评估、导出的开关初始为 `False`，按需打开，完成后关闭。

操作与提交要求见外层 [README](../README.md)、[作业说明](../作业说明.md)；整体方法见 [AlphaZero 学习讲义](参考文献/AlphaZero学习参考.md)。


### 环境准备（已提供）

运行本格，核对目录与 Python 环境。若找不到 `starter/`，打开本版本文件夹并重启内核；本格无需修改。


In [ ]:
from pathlib import Path
import sys

assignment_dir = Path.cwd().resolve()
starter_dir = assignment_dir / "starter"
if not (starter_dir / "support" / "notebook_tools.py").is_file():
    raise FileNotFoundError("请用 VS Code 打开本 Notebook 版本的整个文件夹，并保留其中的 starter/；然后重启内核。")
if str(starter_dir) not in sys.path:
    sys.path.insert(0, str(starter_dir))

from support.notebook_tools import snapshot_code, load_training, run_script, check_notebook
from support.notebook_view import show_training, compare_results, show_replay
import torch
torch.set_num_threads(1)
print("作业目录：", assignment_dir)
print("Python：", sys.executable)
print("PyTorch：", torch.__version__)
print("辅助程序已找到。默认使用 CPU 单线程。")


## 一、导入辅助函数

规则、节点和数据工具已经提供，直接运行。下面会说明与作业有关的接口。


In [ ]:
import math
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from support.game import new_board, legal_moves, play_move, next_player
from support.game import game_result, result_value, convert_value, encode
from support.runtime import Node, evaluate_position, root_noise, visit_counts
from support.runtime import sample_action, sample_batch, set_seed
from support.runtime import prepare_run, save_model, append_log


### 与课堂拟合函数有什么不同？

| 课堂演示 | 本次作业 |
| --- | --- |
| 坐标 `grid` → 函数值 | 棋盘 `states` → 落子分数与局面价值 |
| `potential(grid)` 直接给出标签 | 搜索给出 `pi`，对局结束后得到 `z` |
| 固定目标函数 | 模型变化后，下一轮生成的数据也随之变化 |

参数更新仍沿用课堂流程。本题新增的是**怎样产生训练数据**；`pi` 是当前搜索的建议，不是预先给定的完美答案。


## 二、网络：同时预测策略和价值

三个棋盘通道分别表示我方、对方和空格。网络共用隐藏层，再分出策略和价值两个输出头。

| 变量 | 形状 | 含义 |
| --- | --- | --- |
| `x` | `[B,3,6,7]` | 一批棋盘 |
| `logits` | `[B,7]` | 七列的原始分数，尚未归一化 |
| `value` | `[B]` | 当前行棋方的预期胜负，范围 `[-1,1]` |


In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3 * 6 * 7, 128)
        self.fc2 = nn.Linear(128, 128)
        self.policy = nn.Linear(128, 7)
        self.value = nn.Linear(128, 1)

    def forward(self, x):
        x = x.reshape(-1, 3 * 6 * 7)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        logits = self.policy(x)
        value = torch.tanh(self.value(x)).squeeze(1)
        return logits, value


### 检查输入与输出

运行下格，检查输入 `[2,3,6,7]` 对应两个输出 `[2,7]`、`[2]`。试着只放一个棋盘，核对两个输出的批次维度。


In [ ]:
demo_network = Network()
demo_board = new_board()
demo_states = torch.stack([encode(demo_board, 1), encode(demo_board, -1)])
demo_network.eval()
with torch.no_grad():
    demo_logits, demo_values = demo_network(demo_states)

print("输入形状：", demo_states.shape)
print("落子分数形状：", demo_logits.shape)
print("价值形状：", demo_values.shape)


## 三、自己实现搜索

MCTS 反复选择落子路径，用网络或真实胜负评价叶节点，再更新沿途统计。本题一次模拟到尚未展开的节点为止，不需要随机下完整盘棋。

**价值统一从节点中 `player` 指定的行棋方看。** `player` 表示接下来行动的一方；棋盘中 `0` 为空，`1` 为红方，`-1` 为黄方，列号为 `0～6`。

| 节点属性 / 方法 | 含义 |
| --- | --- |
| `board`、`player` | 棋盘与行棋方 |
| `children` | 列号 → 子节点的字典，初始为空 |
| `prior` | 父节点选择该分支的先验概率 |
| `visits`、`value_sum` | 访问次数、节点行棋方视角的累计价值，初始为 0 |
| `mean_value(player)` | 转换到指定一方视角的平均价值；未访问时为 0 |

后面使用这些已提供的接口：

| 调用 | 返回值 / 作用 |
| --- | --- |
| `Node(board, player, prior=1.0)` | 创建节点 |
| `legal_moves(board)` | 合法列号列表 |
| `play_move(board, col, player)` | 落子后的新棋盘，原棋盘不变 |
| `next_player(player)` | 另一方 |
| `game_result(board)` | `(done, winner)`；终局赢家为 `1`、`-1`，平局为 `0` |
| `evaluate_position(model, board, player)` | `(priors, value)`；满列概率已置零，价值从 `player` 看 |
| `result_value(winner, player)` | 终局胜负：赢 `+1`、输 `-1`、平 `0` |
| `convert_value(value, from_player, to_player)` | 视角转换：同方不变，异方取负 |


In [ ]:
demo_node = Node(new_board(), player=1)
print("轮到哪方：", demo_node.player)
print("访问次数：", demo_node.visits)
print("尚未扩展的分支：", demo_node.children)
print("另一方：", next_player(demo_node.player))


### 1. 选择：比较子节点

遍历 `node.children.values()`，按下式选择并返回得分最高的**节点对象**：

$$Q+cP\frac{\sqrt{N+1}}{n+1}.$$

- `Q = child.mean_value(node.player)`：父节点行棋方视角的平均价值。
- `P = child.prior`，`N = node.visits`，`n = child.visits`，`c = c_puct`。

第一项利用已有评价，第二项鼓励探索先验较高、访问较少的分支。可用循环记录最大值；得分可能全为负，初值可用 `-math.inf`，平方根用 `math.sqrt`。

**思考：**`c_puct=0` 时会怎样？为什么 `Q` 要使用父节点的视角？


In [ ]:
def select_child(node, c_puct):
    """选择 PUCT 得分最高的子节点，返回该 Node。"""
    # TODO 1：遍历 node.children.values()，比较每个子节点的得分。
    # child.mean_value(node.player) 已把价值换成父节点行棋方的视角。
    # PUCT 公式见上方说明；平分时选择其中一个即可。
    raise NotImplementedError("请补全 select_child")


### 2. 扩展：建立合法分支

对每个合法列，从同一个父棋盘出发落子，创建轮到对方行动的节点。先验取 `priors[col]`，节点保存到 `node.children[col]`；函数无需返回值。

**自查：**空棋盘应得到 7 个子节点，每个只有一枚棋子；原棋盘仍为空。


In [ ]:
def expand(node, priors):
    """为合法动作建立子节点，保存到 node.children[列号]。"""
    # TODO 2：调用合法动作、落子、换手函数，为每个动作创建 Node。
    # 子节点的 prior 使用 priors[列号]。play_move 不会改变原棋盘。
    raise NotImplementedError("请补全 expand")


### 3. 回传：累加搜索统计

`path` 包含根到叶的全部节点，传入的 `value` 属于叶节点 `path[-1].player`。

逐个节点用 `convert_value` 转换视角，将结果累加到 `value_sum`，并将 `visits` 增加 1。不要覆盖旧统计，也不要额外再翻转一次符号。

例如路径行棋方为红、黄、红，叶价值为 `+0.6`，应依次累加 `+0.6、-0.6、+0.6`。

**这里更新的是树的统计量，不是梯度，也不修改网络参数。**


In [ ]:
def backpropagate(path, value):
    """更新路径上每个节点的 visits 与 value_sum。"""
    # TODO 3：value 是 path[-1].player 的价值。
    # 用 convert_value(value, 原视角, 新视角) 换成各节点自己的视角。
    raise NotImplementedError("请补全 backpropagate")


### 4. 搜索循环

根节点已建立并扩展。每次模拟依次完成：

1. 从 `root` 重新开始，将它放入新路径。
2. 沿已有子节点调用 `select_child`，记录每个到达的节点，直到叶节点。
3. 用 `game_result` 判断终局：结束则用 `result_value`；否则用网络评价并扩展叶节点。
4. 将路径和叶节点视角的价值交给 `backpropagate`。刚扩展但尚未走入的子节点不在本次路径中。

**终局视角：**红方刚落子获胜，叶节点仍记作下一步轮到黄方，因此价值为 `-1`；转换到红方视角才是 `+1`。

全部模拟完成后再返回。七列访问次数之和应为 `simulations`。


In [ ]:
def search(board, player, model, simulations=32, c_puct=1.0, add_noise=False):
    """返回长度为 7 的数组，表示各列在本次搜索中被访问的次数。"""
    if simulations < 1:
        raise ValueError("搜索次数至少为 1")
    if game_result(board)[0]:
        raise ValueError("已经结束的对局不能继续搜索")

    # 建立根节点。训练时加入少量噪声，让自我对弈探索不同走法。
    root = Node(board, player)
    priors, value = evaluate_position(model, board, player)
    if add_noise:
        priors = root_noise(priors, board)
    expand(root, priors)

    # TODO 4：重复 simulations 次模拟，每次从 root 开始。
    # 选择：沿已有子节点向下走，记录经过的 path。
    # 评估：终局用实际胜负；非终局调用网络，并扩展叶节点。
    # 回传：把叶节点视角的 value 沿 path 更新。
    raise NotImplementedError("请补全 search 中的模拟循环")

    return visit_counts(root)


### 检查搜索

完成四个函数并运行定义格，再执行下面检查；此时不必先写训练部分。根据报错修改、重跑定义与检查，直到看到“本阶段检查通过”。

工具自动保存当前代码用于检查，无需手动管理目录；通过检查不代表模型已有棋力。


In [ ]:
check_notebook(starter_dir, assignment_dir, globals(), stage="search")


## 四、生成自我对弈数据

双方共用一个网络。每步搜索后，**先记录局面，再落子**；整盘结束后才知道胜负标签。

| 记录 | 内容 |
| --- | --- |
| `state = encode(board, player)` | 落子前、当前行棋方视角的棋盘，形状 `[3,6,7]` |
| `pi` | 七列访问次数归一化后的分布，和为 1 |
| `history` 中一条记录 | `(state, pi, 当时行棋方)` |
| `examples` 中一条样本 | `(state, pi, z)` |

补全 `self_play` 的两处：

- **A：记录。** 将 `visits` 除以总次数得到 `pi`，用 `sample_action(pi)` 采样动作，将局面、分布和行棋方放入历史，再执行落子。
- **B：标注。** 对局结束后遍历历史，用 `result_value(winner, 当时行棋方)` 生成 `z`，组成训练样本。

`pi` 来自搜索，不是网络分数；保留完整分布，不再做 softmax 或只保留最大项。红方获胜时，红方记录标 `+1`、黄方记录标 `-1`；平局全部为 `0`。

**思考：**为什么不能用最后一手的 `player` 给所有记录标注？


In [ ]:
def self_play(model, simulations):
    """让模型下完一盘棋，返回用于训练的 (局面, 落子分布, 胜负) 列表。"""
    board = new_board()
    player = 1
    history = []

    while True:
        visits = search(board, player, model, simulations, add_noise=True)

        # TODO A：将访问次数归一化为 pi，调用 sample_action(pi) 选 action。
        # 在落子前记录 (encode(board, player), pi, player) 到 history。
        raise NotImplementedError("请补全 self_play 中的落子与记录")

        board = play_move(board, action, player)
        done, winner = game_result(board)
        if done:
            break
        player = next_player(player)

    examples = []
    # TODO B：根据 winner，为 history 中每个局面生成对应视角的胜负 z。
    # 用 result_value 完成转换，将 (state, pi, z) 放入 examples。
    raise NotImplementedError("请补全 self_play 中的胜负标签")
    return examples


## 五、策略损失与价值损失

策略头学习搜索分布，价值头学习最终胜负：

$$
L_{\mathrm{policy}}=-\frac1B\sum_{i=1}^{B}\sum_{a=0}^{6}\pi_{i,a}\log p_{i,a},
\qquad
L_{\mathrm{value}}=\frac1B\sum_{i=1}^{B}(v_i-z_i)^2.
$$

`B` 为批大小，`a` 为列号；`p、v` 是网络预测，`pi、z` 是收集好的标签。本次更新只对网络参数求梯度，不对搜索过程求梯度。

价值项沿用课堂的均方误差；策略项先对七列求和，再对批次平均。用 `torch.log_softmax(logits, dim=1)` 得到对数概率，最后返回两项之和。

**检查形状：**`logits、target_pi` 为 `[B,7]`，`values、target_z` 为 `[B]`。返回可求梯度的标量 Tensor，勿提前使用 `.item()` 或 `.detach()`。


### 小练习：按列求和，再按样本平均

下面每行是一个样本。改变某个分数，观察概率如何变化；再比较 `sum(dim=1).mean()` 与直接 `.mean()` 的结果，判断哪种对应上面的策略损失。


In [ ]:
demo_scores = torch.tensor([[1.0, 2.0, 3.0], [3.0, 1.0, 1.0]])
demo_probs = torch.softmax(demo_scores, dim=1)
demo_log_probs = torch.log_softmax(demo_scores, dim=1)
print("每行概率：", demo_probs)
print("每行概率和，应接近 1：", demo_probs.sum(dim=1))
print("每行对数概率：", demo_log_probs)

demo_terms = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
demo_row_sums = demo_terms.sum(dim=1)
print("按行相加，应为 6 和 15：", demo_row_sums)
print("再对两行平均：", demo_row_sums.mean())
print("直接平均全部六项：", demo_terms.mean())


In [ ]:
def compute_loss(model, states, target_pi, target_z):
    """策略头学习搜索分布，价值头学习对局结果。"""
    logits, values = model(states)
    # TODO C：按上方公式计算策略损失和价值损失，返回两者之和。
    # target_pi 的形状为 [批大小, 7]；target_z 为 [批大小]。
    # 提示：torch.log_softmax、sum、mean。不要把 pi 变成单个列号。
    raise NotImplementedError("请补全 compute_loss")


## 六、检查数据与损失

完成 A、B、C，重新运行 `self_play`、`compute_loss`，再执行检查。它会核对记录时机、双方标签、损失数值和梯度；通过后再训练。


In [ ]:
check_notebook(starter_dir, assignment_dir, globals(), stage="training")


### 看一盘数据

打开 `TRY_ONE_GAME`，用未训练网络观察：访问次数合计应为 8；样本数不超过 42；输入形状为 `[3,6,7]`，`pi` 之和为 1，`z` 只取 `-1、0、1`。

**试一试：**把两处搜索次数从 8 改为 16，比较时间和分布。为什么一盘非平局对弈会出现正、负两种标签？


In [ ]:
TRY_ONE_GAME = False

if TRY_ONE_GAME:
    try:
        set_seed(2026)
        probe_model = Network()
        counts = search(new_board(), 1, probe_model, simulations=8)
        print("七列访问次数：", counts, "合计：", counts.sum())
        examples = self_play(probe_model, simulations=8)
        print("本盘样本数：", len(examples))
        print("第一条输入形状：", examples[0][0].shape)
        print("第一条搜索分布：", examples[0][1])
        print("各条胜负标签：", [item[2] for item in examples])
    except NotImplementedError as error:
        print("还需完成：", error)
else:
    print("完成核心代码后，可打开 TRY_ONE_GAME 观察一盘数据。")


## 七、先跑一个小实验

默认 **2 轮 × 每轮 2 局，每步搜索 8 次，每轮更新 2 次**，只用于跑通流程。

| 参数 | 含义 |
| --- | --- |
| `ITERATIONS` | 自我对弈与训练重复的轮数 |
| `GAMES_PER_ITERATION` | 每轮对局数 |
| `SIMULATIONS` | 每步落子的搜索次数 |
| `BATCH_SIZE` | 每次更新抽取的样本数 |
| `TRAIN_STEPS` | 每轮参数更新次数 |
| `MEMORY_SIZE` | 最多保留的近期样本数 |
| `LEARNING_RATE` | 学习率 |

每轮下棋后更新 `TRAIN_STEPS` 次，默认共更新 `2×2=4` 次。`SIMULATIONS` 是每步搜索次数，与参数更新次数不同。

先用 CPU 跑通，再逐项增加规模，例如 10 轮、每轮 20 局、搜索 32 次、更新 20 次。本实现含较多 Python 搜索，GPU 未必更快，可按课堂方法比较用时与资源占用。

新实验换 `RUN_NAME` 并重新运行参数格；每次从随机网络开始，不自动续训。


In [ ]:
RUN_NAME = "trial01"
ITERATIONS = 2
GAMES_PER_ITERATION = 2
SIMULATIONS = 8
BATCH_SIZE = 16
TRAIN_STEPS = 2
MEMORY_SIZE = 10000
LEARNING_RATE = 0.001
DEVICE = "cpu"

RUN_DIR = assignment_dir / "notebook_runs" / RUN_NAME
print("本次实验目录：", RUN_DIR)


### 主训练循环（已提供）

每轮先生成对局，再从经验池抽样更新：

| 代码 | 作用 |
| --- | --- |
| `replay.extend(...)` | 加入一盘的全部样本 |
| `replay[-MEMORY_SIZE:]` | 保留最近的样本 |
| `sample_batch(...)` | 一起抽取 `states、target_pi、target_z`，三者保持对应；允许重复抽样 |

`task.Network/self_play/compute_loss` 是工具保存的本次实现，保证训练、评估和导出使用同一套代码。每轮保存模型与日志；经验池和损失会随新对局变化，损失不一定单调下降。


In [ ]:
START_TRAINING = False

if START_TRAINING:
    config = {
        "ITERATIONS": ITERATIONS, "GAMES_PER_ITERATION": GAMES_PER_ITERATION,
        "SIMULATIONS": SIMULATIONS, "BATCH_SIZE": BATCH_SIZE,
        "TRAIN_STEPS": TRAIN_STEPS, "MEMORY_SIZE": MEMORY_SIZE,
        "LEARNING_RATE": LEARNING_RATE, "DEVICE": DEVICE,
    }
    src_dir = snapshot_code(starter_dir, RUN_DIR, globals(), config=config)
    run_script(src_dir, "check.py", ["training"])
    task = load_training(src_dir)

    set_seed(2026)
    model = task.Network().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    save_model(model, RUN_DIR / "initial.pt")
    replay = []
    start = time.time()

    # 每轮先生成数据，再更新参数。
    for iteration in range(1, ITERATIONS + 1):
        model.eval()
        for game in range(GAMES_PER_ITERATION):
            replay.extend(task.self_play(model, SIMULATIONS))
            print("轮次", iteration, "对局", game + 1, "/", GAMES_PER_ITERATION, flush=True)
        replay = replay[-MEMORY_SIZE:]

        model.train()
        for step in range(TRAIN_STEPS):
            states, target_pi, target_z = sample_batch(replay, BATCH_SIZE, DEVICE)
            loss = task.compute_loss(model, states, target_pi, target_z)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        save_model(model, RUN_DIR / "model.pt")
        row = {"iteration": iteration, "positions": len(replay),
               "loss": loss.item(), "seconds": round(time.time() - start, 1)}
        append_log(RUN_DIR / "log.jsonl", row)
        print(row, flush=True)
else:
    print("准备好后把 START_TRAINING 改为 True，运行本格。")


### 运行遇到问题

| 情况 | 处理 |
| --- | --- |
| 尚未补全 / 名字未定义 | 补全函数并重新运行定义；缺少变量时补跑准备格 |
| 训练目录已存在 | 换 `RUN_NAME` 后重跑参数格；检查失败也可能已创建目录 |
| 一段时间没有输出 | 每盘结束后才打印进度；先减小参数，仍不结束则检查搜索循环 |
| 中断训练 | 已保存轮次保留，未保存进度丢失；再次运行不自动续训 |


## 八、查看训练结果

下面读取日志，画出损失和经验池样本数。小实验只有两个点；判断棋力还需对战评估。


In [ ]:
if (RUN_DIR / "log.jsonl").exists():
    show_training(RUN_DIR / "log.jsonl")
else:
    print("还没有训练日志；训练完成一轮后再看曲线。")


### 比较训练前后

两个模型使用相同搜索次数和种子，对战随机合法策略，先后手各半。默认 4 局用于检查流程，正式比较可设 `EVAL_GAMES≥40`。

评估对象是“网络 + 搜索”，随机网络也可能靠搜索获胜。`EVAL_SIMULATIONS` 单独控制评估搜索次数，默认 32。

评估读取本次保存的源码与模型。再次评估换 `EVAL_NAME`，避免覆盖。


In [ ]:
RUN_EVALUATION = False
EVAL_NAME = "test01"
EVAL_GAMES = 4              # 至少 2 的偶数
EVAL_SIMULATIONS = 32
BEFORE = RUN_DIR / (EVAL_NAME + "-before.json")
AFTER = RUN_DIR / (EVAL_NAME + "-after.json")

if RUN_EVALUATION:
    if BEFORE.exists() or AFTER.exists():
        raise FileExistsError("已有同名结果，请修改 EVAL_NAME。")
    for weights, output in [("initial.pt", BEFORE), ("model.pt", AFTER)]:
        run_script(RUN_DIR / "src", "evaluate.py", [
            str(RUN_DIR / weights), "--games", str(EVAL_GAMES),
            "--simulations", str(EVAL_SIMULATIONS), "--seed", "2026",
            "--output", str(output),
        ])
else:
    print("训练完成后，可打开 RUN_EVALUATION 比较训练前后。")


In [ ]:
if BEFORE.exists() and AFTER.exists():
    compare_results(BEFORE, AFTER)
else:
    print("完成训练前后两组评估后，这里会显示统计。")


### 复盘一局

- `GAME_INDEX`：第几局，从 0 开始。
- `MOVE`：已落子数，`0` 为空盘，`None` 为终局。
- `RESULTS`：`AFTER` 看训练后，`BEFORE` 看训练前。

修改后重跑本格。观察模型是否抓住机会或忽略威胁，选一个局面写入实验记录。红方先手，列号为 `0～6`。


In [ ]:
GAME_INDEX = 0
MOVE = None
RESULTS = AFTER             # 改为 BEFORE 查看训练前

if RESULTS.exists():
    show_replay(RESULTS, game_index=GAME_INDEX, move=MOVE)
else:
    print("评估结束后，就有可复盘的棋谱。")


### 下次只查看已有结果

依次运行：**环境准备 → 导入 → 参数格（旧 `RUN_NAME`）→ 曲线 → 评估设置（旧 `EVAL_NAME`，`RUN_EVALUATION=False`）→ 统计 / 复盘。**

无需重跑训练或对战；评估设置格只是恢复结果文件路径。


## 九、导出对战文件

打开下方导出开关，生成 `code.zip` 和 `model.pt`。在服务器上训练后，在 VS Code 左侧导出目录中将这两个文件分别右键下载到本地，再到 [课程平台](https://botlab.aiphys.cn/) 上传，无需解压或改名。

“策略代码”选 `code.zip`，“模型参数”选 `model.pt`，点击“上传并检查”；状态为“检查通过”后即可对战，未通过时查看日志。

导出沿用本次实验保存的源码与模型。修改网络或训练实现后，需重跑定义、换实验名并重新训练；再次导出时换 `EXPORT_NAME`。

网站对战每步默认搜索 32 次，与训练、评估参数分开。若只调整对战搜索次数，修改本次实验的 `notebook_runs/实验名/src/support/bot.py` 中的 `SIMULATIONS`，再重新导出即可，无需重训。


In [ ]:
EXPORT_NOW = False
EXPORT_NAME = "upload"  # 再导出时换一个名称

if EXPORT_NOW:
    run_script(RUN_DIR / "src", "export.py", [
        str(RUN_DIR / "model.pt"), "--out", str(RUN_DIR / EXPORT_NAME),
    ])
else:
    print("确认训练和评估完成后，可打开 EXPORT_NOW 导出。")


## 十、整理与提交

结果位于 `notebook_runs/实验名/`：

| 文件 | 内容 |
| --- | --- |
| `src/`、`config.json` | 本次源码、训练配置 |
| `initial.pt`、`model.pt` | 训练前后参数 |
| `log.jsonl` | 训练日志 |
| `test01-before.json`、`test01-after.json` | 对战统计与棋谱 |
| `upload/` | 网站上传文件 |

按外层 [作业说明](../作业说明.md) 整理提交：将本次 `src/` 复制为 `starter/`，与补全的 Notebook 和模型放在一起。无需再手填 `.py` 模板，网站 `code.zip` 不能代替训练源码。

模型文件不包含经验池和完整续训状态。结束后保存 Notebook，并将运行开关改回 `False`。


<details>
<summary>选读：自动整理代码的范围</summary>

工具收集已运行的 `Network` 和六个核心函数。直接使用准备格导入的接口及 `math、np、torch、nn、F`；不要在核心函数内重新导入本作业模块。常量放在函数内或参数中，类属性放在 `__init__` 中。

新增顶层函数或依赖时，需要相应扩展整理工具；按原骨架补全无需处理。生成的 `src/train.py` 可单独运行，结果写入同级 `rerun/`。

</details>
